# Lesson 2.3: Combining Events: OR and AND

Check the addition rule (heart or king) and the multiplication rule for independent events (heads and a 6) by exact counting and by simulation.

Part of *Statistics from Scratch*. Run each cell with **Shift + Enter**, top to bottom.

## The question

A card game uses an ordinary 52-card deck. A player asks: when I draw one card, what is the chance it is a heart or a king? A second player asks a different question: when I flip a coin and roll a die, what is the chance of heads and a 6? Can the addition and multiplication rules answer both, and can a simulation confirm them?

If a package is missing on your computer, run the next cell once. In Google Colab everything is already installed.

In [ ]:
%pip install -q numpy scipy pandas statsmodels matplotlib

## Step 1: Build a deck and count

We number the 52 cards 0 to 51: the first 13 numbers are hearts, and the last card in each group of 13 is a king. Counting gives the pieces of the addition rule, including the king of hearts that is counted twice.

In [1]:
import numpy as np  # Load NumPy, which lets us do arithmetic on a whole list of cards at once.

cards = np.arange(52)  # Number the 52 cards 0 to 51 so each card has an ID.
heart = cards // 13 == 0  # Mark the hearts: cards 0 to 12 are the first group of 13, which we call hearts.
king = cards % 13 == 12  # Mark the kings: the last card in each group of 13 (number 12 in its group) is a king.

print('hearts:', heart.sum(), ' kings:', king.sum(), ' both:', (heart & king).sum())  # Count hearts, kings and cards that are both. Summing True/False values counts the Trues.
print('heart or king, counted directly:', (heart | king).sum(), 'of 52')  # Count cards that are a heart or a king directly, as the answer we will test the rule against.

hearts: 13  kings: 4  both: 1
heart or king, counted directly: 16 of 52


**What this means.** There are 13 hearts, 4 kings and 1 card that is both (the king of hearts). Counting directly, 16 of the 52 cards are a heart or a king, not 17, because the king of hearts would otherwise be counted twice.

## Step 2: The addition rule

P(A or B) = P(A) + P(B) - P(A and B). Subtracting the overlap fixes the double count, and the answer matches the direct count.

In [2]:
p_heart = heart.mean()  # Turn each count into a probability: the mean of True/False values is the proportion of Trues. Start with hearts.
p_king = king.mean()  # The same for kings.
p_both = (heart & king).mean()  # The same for cards that are both.

print(f'P(heart) + P(king) = {p_heart + p_king:.4f}  (too big)')  # Add the two probabilities without the overlap fix. This is too big because the king of hearts is counted twice.
print(f'minus overlap {p_both:.4f}   = {p_heart + p_king - p_both:.4f}')  # Subtract the overlap, which is the addition rule P(A) + P(B) - P(A and B).
print(f'direct count 16/52    = {(heart | king).mean():.4f}')  # Compare with the direct count of heart-or-king cards.

P(heart) + P(king) = 0.3269  (too big)
minus overlap 0.0192   = 0.3077
direct count 16/52    = 0.3077


**What this means.** Adding P(heart) and P(king) gives 0.3269, which is too big. Subtracting the overlap 0.0192 gives 0.3077, and the direct count 16/52 is also 0.3077. The addition rule works because it removes the double-counted king of hearts.

## Step 3: Check the addition rule by simulation

Now we really draw cards, one at a time with replacement, using a seeded generator. The share of draws that are a heart or a king should land close to 16/52.

In [3]:
rng = np.random.default_rng(42)  # Create a seeded generator so the simulated draws can be repeated.
draws = rng.integers(0, 52, size=200_000)  # Draw 200,000 random cards, with replacement, as numbers 0 to 51.

sim_or = ((draws // 13 == 0) | (draws % 13 == 12)).mean()  # Work out the share of draws that are a heart or a king, using the same test as before.
print(f'simulated P(heart or king) = {sim_or:.4f}')  # Print the simulated probability.
print(f'exact 16/52                = {16 / 52:.4f}')  # Print the exact 16/52 beside it for comparison.

simulated P(heart or king) = 0.3068
exact 16/52                = 0.3077


**What this means.** The simulated probability of a heart or a king was 0.3068, close to the exact 16/52 = 0.3077. They differ in the third decimal because a simulation has random error, which shrinks as the number of draws grows.

## Step 4: The multiplication rule

A coin and a die do not affect each other, so the events are independent and P(heads and 6) = 1/2 x 1/6. We flip and roll 200,000 times and count how often both happen.

In [4]:
coin = rng.integers(0, 2, size=200_000)  # Flip 200,000 coins (0 or 1). A coin does not know what the die will show.
die = rng.integers(1, 7, size=200_000)  # Roll 200,000 dice with values 1 to 6.

both = ((coin == 1) & (die == 6)).mean()  # Find the share of trials where the coin shows heads (1) and the die shows 6.
print(f'simulated P(heads and 6) = {both:.4f}')  # Print the simulated probability of both.
print(f'1/2 x 1/6                = {0.5 * 1 / 6:.4f}')  # Print the rule 1/2 x 1/6, which is correct because the events are independent.
print(f'P(heads) x P(6) in sample = {(coin == 1).mean() * (die == 6).mean():.4f}')  # Multiply the two separate shares in the same sample, to show that the product rule also holds there.

simulated P(heads and 6) = 0.0830
1/2 x 1/6                = 0.0833
P(heads) x P(6) in sample = 0.0826


**What this means.** Heads and a 6 happened in 0.0830 of the trials, close to 1/2 x 1/6 = 0.0833. The product of the separate shares in the sample was 0.0826. Multiplying works here because the coin and the die do not affect each other.

## So, what is the answer?

The chance of drawing a heart or a king is 16/52 = 0.3077, found by the addition rule (0.3269 minus the 0.0192 overlap) and confirmed by counting and simulation. The chance of heads and a 6 is 1/12 = 0.0833 because the events are independent. The caution: multiplying is only valid for independent events, and adding without subtracting the overlap overcounts.

## Now you try

A game show rolls one fair six-sided die. The host calls the roll a win if it is even or greater than 4. In a second round, two different players each roll a die, and the host asks how likely it is that both roll a 6.

**Data:** Die faces: [1, 2, 3, 4, 5, 6]. Events: even = {2, 4, 6}; big = faces above 4. For the simulation use the seed 3 and 200,000 rolls of each of two dice.

1. Count the even faces, the faces above 4, and the faces that are both.
2. Find P(even or above 4) with the addition rule and by counting.
3. Find P(both dice show 6) with the multiplication rule and by simulation.

*Hint:* Use P(A or B) = P(A) + P(B) - P(A and B), and remember that the king-of-hearts style overlap here is the face that is both even and above 4. Two dice are independent, so multiply.

In [ ]:
# Your turn: write your code here


### Solution (try first, then peek)

In [ ]:
import numpy as np

die = np.arange(1, 7)
even = die % 2 == 0
big = die > 4
print('even:', even.sum(), ' >4:', big.sum(), ' both:', (even & big).sum())
print(f'addition rule: {even.mean():.4f} + {big.mean():.4f} - {(even & big).mean():.4f} = {(even | big).mean():.4f}')

rng = np.random.default_rng(3)
a = rng.integers(1, 7, size=200_000)
b = rng.integers(1, 7, size=200_000)
print(f'simulated P(even or >4)    = {((a % 2 == 0) | (a > 4)).mean():.4f}')
print(f'simulated P(both dice 6)   = {((a == 6) & (b == 6)).mean():.4f}  (exact {1/36:.4f})')

even: 3  >4: 2  both: 1
addition rule: 0.5000 + 0.3333 - 0.1667 = 0.6667
simulated P(even or >4)    = 0.6654
simulated P(both dice 6)   = 0.0275  (exact 0.0278)


**Answer.** There are 3 even faces, 2 faces above 4 and 1 that is both (the 6). The addition rule gives 0.5000 + 0.3333 - 0.1667 = 0.6667, so 2/3 of rolls win. For two dice, the rule gives 1/36 = 0.0278 and the simulation found 0.0275, close to it. The first answer subtracts the overlap; the second multiplies because the dice are independent.

## The same question in R, SPSS and Stata

**R**

```r
card <- 0:51  # Number the cards 0 to 51.
heart <- card %/% 13 == 0  # Mark the hearts: integer division by 13 gives 0 for the first group.
king  <- card %% 13 == 12  # Mark the kings: the remainder 12 is the last card in each group.
mean(heart) + mean(king) - mean(heart & king)  # Addition rule: P(heart) + P(king) - P(both).
mean(heart | king)            # direct count  # Direct count of heart-or-king cards; | means 'or'.

0.5 * 1/6                      # heads and a 6  # Multiplication rule for independent events: 1/2 x 1/6.

set.seed(42)  # Fix the seed for repeatable random numbers.
n <- 200000  # Use 200,000 trials.
mean(sample(0:1, n, TRUE) == 1 & sample(6, n, TRUE) == 6)  # Simulate coin and die, and take the share of trials where heads and a 6 occur together.
```

How to read it: Check that the two card lines both print 0.3077, which shows the addition rule agrees with counting. The next line prints 0.0833. The simulated value near 0.0833 differs from Python's in the third decimal because R's random numbers differ.

**SPSS** (Transform > Compute Variable; Analyze > Descriptive Statistics > Descriptives)

```
INPUT PROGRAM.
LOOP #i = 0 TO 51.
COMPUTE card = #i.
END CASE.
END LOOP.
END FILE.
END INPUT PROGRAM.
COMPUTE heart = (TRUNC(card/13) = 0).
COMPUTE king = (MOD(card, 13) = 12).
COMPUTE both = (heart = 1 AND king = 1).
COMPUTE either = (heart = 1 OR king = 1).
DESCRIPTIVES VARIABLES=heart king both either.
```

How to read it: In the Descriptive Statistics table, read the Mean column: heart .25, king .08, both .02 and either .31 (the exact values are 0.25, 0.0769, 0.0192 and 0.3077). Check that .25 + .08 - .02 matches .31.

**Stata**

```
clear
set obs 52
gen card = _n - 1
gen heart = floor(card/13) == 0
gen king = mod(card, 13) == 12
gen either = heart | king
summarize heart king either
display 0.5 * 1/6

clear
set seed 42
set obs 200000
gen win = (runiform() < 0.5) & (ceil(6*runiform()) == 6)
summarize win
```

How to read it: In the first summarize, the means are .25 (hearts), .0769231 (kings) and .3076923 (either). The display line prints .0833. The last mean is the simulated chance of heads and a 6, close to .0833 but not equal to Python's 0.0830.